# Lecture: Building a Reproducible Analysis Workflow

Use three Pokémon questions to practice a reproducible analysis. We will predict results, introduce Pandas operations as they are needed, and connect each conclusion to checked evidence.


## Learning goals

- Organize an analysis using the six-step workflow.
- Identify observations and relevant features.
- Group records and compare means with group counts.
- Use vectorized comparisons and numeric indicators to calculate percentages.
- Distinguish absent secondary types from unknown data using documentation.
- Check results and state conclusions and limitations.


## The reproducible analysis workflow

We will use six steps:

1. **Question:** What do we want to learn?
2. **Data:** Which observations and features can help answer it?
3. **Evidence:** What should we calculate and display to answer the question?
4. **Check:** Does the result make sense in context? Do a few original records support the meaning of the calculation?
5. **Conclusion:** What claim is supported?
6. **Limitation:** What should we avoid concluding?


Code that runs without an error can still answer the wrong question. A reproducible analysis preserves the code and records how each result supports the conclusion.


# Part 1: Pokémon base stats


## Background

Use Alberto Barradas's [Pokémon with stats dataset on Kaggle](https://www.kaggle.com/datasets/abcsds/pokemon). A fixed copy is supplied as `data/pokemon_stats.csv`, so everyone analyzes the same records without needing a Kaggle account.

The file has 800 records representing Pokémon and alternate forms from Generations 1–6, with 721 distinct Pokédex numbers. One row is a listed Pokémon/form, not a battle or an individual player's Pokémon. Alternate forms can share a Pokédex number; do not assume repeated numbers are accidental duplicates. This historical file does not cover every current generation.

Base stats are game attributes. `Total` is the sum of six base stats, not a win rate or a guarantee of battle success. No Pokémon expertise is required: use the dictionary and definitions below.


### Data dictionary

| Feature | Description |
| --- | --- |
| `#` | Pokédex number; an identifier that can repeat for alternate forms |
| `Name` | Name of the listed Pokémon/form |
| `Type 1` | Primary type in this file |
| `Type 2` | Secondary type; blank for records with no secondary type |
| `Total` | Sum of HP, Attack, Defense, Sp. Atk, Sp. Def, and Speed |
| `HP` | Base hit points |
| `Attack` | Base physical attack stat |
| `Defense` | Base physical defense stat |
| `Sp. Atk` | Base special attack stat |
| `Sp. Def` | Base special defense stat |
| `Speed` | Base speed stat |
| `Generation` | Generation of introduction recorded in this file |
| `Legendary` | True/False classification supplied by the dataset author |

Use the dataset's classifications as recorded. Primary type alone does not capture all type combinations or battle matchups.


## Scope for all three investigations

Use all 800 records, including Legendary Pokémon. Count alternate forms as individual Pokémon for this lecture: each row receives equal weight, and repeated Pokédex numbers are retained. Use the generation labels recorded in the file. These questions concern this historical dataset, not every Pokémon released since it was assembled.


## Investigation 1: Generation and average HP

### Step 1: Question

**Which generation of Pokémon (including alternative forms as individual Pokémon) has the highest average HP?**

Use the arithmetic mean as the average.


### Prediction — before calculating


**Which generation do you predict will have the highest average HP, and why? Record your prediction before running the analysis.**


Your prediction:


### Step 2: Data


Import Pandas as `pd`, read `data/pokemon_stats.csv` into `pokemon_df`, and display its first and last five rows, shape, `.info()`, and missing-value counts.


In [1]:
import pandas as pd
pokemon_df = pd.read_csv('data/pokemon_stats.csv')
pokemon_df.head()

,#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,1,Bulbasaur,Grass,Poison,318,45,49,49,65,65,45,1,False
1,2,Ivysaur,Grass,Poison,405,60,62,63,80,80,60,1,False
2,3,Venusaur,Grass,Poison,525,80,82,83,100,100,80,1,False
3,3,VenusaurMega Venusaur,Grass,Poison,625,80,100,123,122,120,80,1,False
4,4,Charmander,Fire,NaN,309,39,52,43,60,50,65,1,False


In [2]:
pokemon_df.tail()

,#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
795,719,Diancie,Rock,Fairy,600,50,100,150,100,150,50,6,True
796,719,DiancieMega Diancie,Rock,Fairy,700,50,160,110,160,110,110,6,True
797,720,HoopaHoopa Confined,Psychic,Ghost,600,80,110,60,150,130,70,6,True
798,720,HoopaHoopa Unbound,Psychic,Dark,680,80,160,60,170,130,80,6,True
799,721,Volcanion,Fire,Water,600,80,110,120,130,90,70,6,True


In [3]:
pokemon_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   #           800 non-null    int64
 1   Name        800 non-null    str  
 2   Type 1      800 non-null    str  
 3   Type 2      414 non-null    str  
 4   Total       800 non-null    int64
 5   HP          800 non-null    int64
 6   Attack      800 non-null    int64
 7   Defense     800 non-null    int64
 8   Sp. Atk     800 non-null    int64
 9   Sp. Def     800 non-null    int64
 10  Speed       800 non-null    int64
 11  Generation  800 non-null    int64
 12  Legendary   800 non-null    bool 
dtypes: bool(1), int64(9), str(3)
memory usage: 89.3 KB


In [4]:
pokemon_df.isnull().sum()

#               0
Name            0
Type 1          0
Type 2        386
Total           0
HP              0
Attack          0
Defense         0
Sp. Atk         0
Sp. Def         0
Speed           0
Generation      0
Legendary       0
dtype: int64

**What does one row represent, which features answer this question, and are those features complete?**


### Step 3: Evidence


`groupby()` partitions the rows by generation. `.agg(["count", "mean"])` displays the number of HP observations and their mean for each generation. Read the output directly to compare generations; we do not need to save a separate summary DataFrame.


Group `pokemon_df` by `Generation` and summarize `HP` using `.agg(["count", "mean"])`. Display the result directly, then identify the highest mean and its group count.


In [7]:
pokemon_df.groupby("Generation")["HP"].agg(["count", "mean"])

,count,mean
Generation,,
1,166,65.819277
2,106,71.207547
3,160,66.543750
4,121,73.082645
5,165,71.787879
6,82,68.268293


Sort the summary by its `mean` column from largest to smallest. `ascending=False` requests descending order. The original Pokémon DataFrame is unchanged.


In [8]:
pokemon_df.groupby("Generation")["HP"].agg(["count", "mean"]).sort_values("mean", ascending=False)

,count,mean
Generation,,
4,121,73.082645
5,165,71.787879
2,106,71.207547
6,82,68.268293
3,160,66.543750
1,166,65.819277


**Which generation has the highest mean HP? Report its mean and number of records.**


### Step 4: Check


**What mean HP values would make you suspicious about the data quality because they wouldn't make sense?**


### Step 5: Conclusion


**Write a sentence answering the question, then explain whether the evidence supports your prediction.**


### Step 6: Limitation


**Does it mean that every Gen 4 Pokémon has a higher HP than every Pokémon in other generations?**


## Investigation 2: Secondary types

### Step 1: Question

**What percent of Pokémon (including alternative forms as individual Pokémon) have a secondary type?**


**Before calculating, do you predict that fewer than half, about half, or more than half have a secondary type? Explain briefly.**


Your prediction:


### Step 2: Data

Reuse all rows in `pokemon_df` and the `Type 2` feature. According to the dictionary, a blank secondary type means no secondary type in this file. Retain those rows in the denominator; they are part of the question.


### Step 3: Evidence


`.notna()` produces one Boolean per row. `.astype(int)` converts True to 1 and False to 0. Summing a 0/1 indicator counts the records with a secondary type. Dividing that sum by the length of the indicator gives the fraction of all records; multiplying by 100 converts it to a percentage. Assigning a Series to a new column aligns it with the DataFrame rows.


Create `has_secondary_type` directly in `pokemon_df` using `.notna().astype(int)` on `Type 2`. Calculate `secondary_type_count` by summing the indicator and `secondary_type_percent` by dividing that sum by `len(pokemon_df["has_secondary_type"])` and multiplying by 100.


In [9]:
pokemon_df["Type 2"].value_counts()

Type 2
Flying      97
Ground      35
Poison      34
Psychic     33
Fighting    26
Grass       25
Fairy       23
Steel       22
Dark        20
Dragon      18
Ice         14
Rock        14
Water       14
Ghost       14
Fire        12
Electric     6
Normal       4
Bug          3
Name: count, dtype: int64

In [13]:
pokemon_df.notna().astype("int")

,#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,1,1,1,1,1,1,1,1,1,1,1,1,1
1,1,1,1,1,1,1,1,1,1,1,1,1,1
2,1,1,1,1,1,1,1,1,1,1,1,1,1
3,1,1,1,1,1,1,1,1,1,1,1,1,1
4,1,1,1,0,1,1,1,1,1,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,1,1,1,1,1,1,1,1,1,1,1,1,1
796,1,1,1,1,1,1,1,1,1,1,1,1,1
797,1,1,1,1,1,1,1,1,1,1,1,1,1
798,1,1,1,1,1,1,1,1,1,1,1,1,1


In [15]:
pokemon_df["has_secondary_type"] = pokemon_df["Type 2"].notna().astype("int")

pokemon_df["has_secondary_type"].head()

pokemon_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   #                   800 non-null    int64
 1   Name                800 non-null    str  
 2   Type 1              800 non-null    str  
 3   Type 2              414 non-null    str  
 4   Total               800 non-null    int64
 5   HP                  800 non-null    int64
 6   Attack              800 non-null    int64
 7   Defense             800 non-null    int64
 8   Sp. Atk             800 non-null    int64
 9   Sp. Def             800 non-null    int64
 10  Speed               800 non-null    int64
 11  Generation          800 non-null    int64
 12  Legendary           800 non-null    bool 
 13  has_secondary_type  800 non-null    int64
dtypes: bool(1), int64(10), str(3)
memory usage: 95.6 KB


Display the numerator, denominator, and percentage rounded to two decimal places.


In [16]:
secondary_type_count = pokemon_df["has_secondary_type"].sum()
secondary_type_count

np.int64(414)

In [17]:
secondary_type_percent = secondary_type_count/ len(pokemon_df["has_secondary_type"]) * 100
secondary_type_percent

np.float64(51.74999999999999)

In [18]:
print( round(secondary_type_percent, 2))

51.75


### Step 4: Check


Read a few original type labels beside the new indicator. For each row, decide what the indicator should be before comparing it with the calculated value. Is the code answering “has a secondary type,” or did we accidentally reverse that meaning?


In [ ]:
pokemon_df[["Name", "Type 1", "Type 2", "has_secondary_type"]].head(10)


**Does it appear that values are being assigned to `has_secondary_type` correctly?**


### Step 5: Conclusion


**Answer the research question in one sentence using both the percentage and counts.**


### Step 6: Limitation


**Why should we consult the dictionary before interpreting blank values?**


## Investigation 3: Attack compared with Defense

### Step 1: Question

**What percent of Pokémon (including alternative forms as individual Pokémon) have an Attack that is higher than, lower than, or equal to their Defense?**

Use the physical `Attack` and `Defense` base stats, not the special stats. In the main analysis, **equal means exactly equal**.


**Before calculating, which group do you predict will be largest? Do you expect many exact ties?**


Your prediction:


### Step 2: Data

Reuse `Attack` and `Defense` from the initial inspection. Both are numeric base-stat measurements. We will compare the two values within each row.


### Step 3: Evidence


Comparing two Series performs an element-by-element comparison for aligned rows. `.loc[condition, "column"]` assigns labels only where the condition is True. With complete numeric values, greater than, less than, and exact equality divide the rows into three non-overlapping groups.


Create three Boolean conditions: `attack_higher`, `attack_lower`, and `attack_equal`. Use `>`, `<`, and `==`. Use `.loc` to assign readable labels for the three conditions. The first assignment creates `attack_defense_group`; no separate initialization is needed. Because Attack and Defense are complete, the three conditions cover every row.


In [19]:
#Conditionals
attack_higher = pokemon_df["Attack"] > pokemon_df["Defense"]

attack_lower = pokemon_df["Attack"] < pokemon_df["Defense"]

attack_same = pokemon_df["Attack"] == pokemon_df["Defense"]

In [23]:
#look in dataframe, find where condtion attack higher is true, create new feature and assign value
pokemon_df.loc[attack_higher, "attack_defense_group"] = "Higher"

In [24]:
pokemon_df.loc[attack_lower, "attack_defense_group"] = "Lower"

In [25]:
pokemon_df.loc[attack_same, "attack_defense_group"] = "Same"

Create `attack_defense_summary_df` with counts from `.value_counts()` and percentages calculated as each count divided by the total number of rows times 100.


`.value_counts()` returns a Series: category labels are its index and counts are its values. `.to_frame("pokemon_count")` converts that Series to a one-column DataFrame and names the column `pokemon_count`. The index and counts stay the same. A DataFrame lets us add a percentage column alongside the counts.


In [26]:
pokemon_df["attack_defense_group"].value_counts()

attack_defense_group
Higher    433
Lower     298
Same       69
Name: count, dtype: int64

In [29]:
pokemon_df["attack_defense_group"].value_counts(normalize=True) * 100

attack_defense_group
Higher    54.125
Lower     37.250
Same       8.625
Name: proportion, dtype: float64

Display the counts and percentages. Round the displayed percentages to two decimal places.


In [30]:
pokemon_df["attack_defense_group"].value_counts(normalize=True).mul(100).round(2)

attack_defense_group
Higher    54.12
Lower     37.25
Same       8.62
Name: proportion, dtype: float64

### Step 4: Check


Calculate `attack_defense_difference` as Attack minus Defense, then use `.describe()` within each assigned group. Focus on the `min` and `max`: Higher should contain only positive differences, Lower only negative differences, and Equal only zeros.


In [31]:
pokemon_df["attack_defense_difference"] = pokemon_df["Attack"] - pokemon_df["Defense"]


In [32]:
pokemon_df.groupby("attack_defense_group")["attack_defense_difference"].describe()


,count,mean,std,min,25%,50%,75%,max
attack_defense_group,,,,,,,,
Higher,433.0,27.711316,21.256857,1.0,12.0,24.0,38.0,160.0
Lower,298.0,-26.416107,25.609894,-220.0,-35.0,-20.0,-10.0,-1.0
Same,69.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0


**Do the minimum and maximum differences match the meaning of Higher, Lower, and Equal? What would make you suspicious?**


### Step 5: Conclusion


**Write a sentence reporting all three percentages and their counts.**


### Step 6: Limitation


**Why does higher Attack than Defense not prove that a Pokémon wins more battles?**


## Summary

We answered three questions using all 800 listed Pokémon/forms. Grouped means compared generations, an indicator measured the share with a secondary type, and vectorized comparisons classified Attack relative to Defense. We inspected the data once, then used targeted sanity checks to examine plausible HP values, indicator meanings, and comparison labels. A sanity check can reveal a mistake without proving every result correct. The lab applies the same workflow to a dataset you choose.
